In [2]:
from embedder import Embedder

embed = Embedder()

2026-06-29 18:16:12.134843286 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
# Question 1
query = 'How does approximate nearest neighbor search work?'
q1 = embed.encode(query)
len(q1)


384

In [4]:
# Answer 1
q1[0]

np.float64(-0.02058203437252893)

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
# Question 2
target = '02-vector-search/lessons/07-sqlitesearch-vector.md'
doc = next(d for d in documents if d['filename'] == target)
content = doc["content"]
content

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [7]:
# Embedding content
q2 = embed.encode(content)

# Calculating cosine similarity
q2.dot(q1)

np.float64(0.36107026789538205)

In [8]:
# Question 3
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
texts = [chunk.get("text", chunk.get("content", "")) if isinstance(chunk, dict)
         else getattr(chunk, "text", getattr(chunk, "content", "")) for chunk in chunks]
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

len(vectors)

X = np.array(vectors)


  0%|          | 0/6 [00:00<?, ?it/s]

In [21]:
scores = X.dot(q1)
scores.max(), scores.argmax(), scores.mean(), scores.std()



(np.float64(0.6489016436447387),
 np.int64(94),
 np.float64(0.09691194290272033),
 np.float64(0.1338636688162658))

In [ ]:
docu

TypeError: list indices must be integers or slices, not str

In [ ]:
# Searching automatically using minsearch
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)